# Notebook 4: Emulator Validation -- Is It Ready to Replace FastChem?

The VULCAN emulator is a neural network trained to reproduce the output of FastChem,
an equilibrium chemistry solver. This notebook tests whether it is accurate and fast
enough to replace FastChem in a real atmospheric retrieval pipeline.

## Four tests

| # | Test | What we check | Pass criterion |
|---|------|---------------|----------------|
| 1 | **Fidelity** | Does the emulator reproduce FastChem VMR profiles? | Median error < 0.15 dex (all species), < 0.05 dex (CO), < 0.10 dex (H2O) |
| 2 | **Gradients** | Do derivatives flow through the emulator? | `jax.grad` finite, `jacfwd ~ jacrev`, `vmap` works |
| 3 | **Speed** | How much faster is the emulator? | > 10x speedup over FastChem subprocess |
| 4 | **Retrieval** | Same posterior as FastChem? | Emulator and FastChem 1-sigma intervals overlap for all parameters |

## The key idea behind Test 4: apples-to-apples retrieval

'Retrieval' means: given a synthetic spectrum computed with FastChem at known
'truth' parameters, can a sampler recover those parameters?

We use **emcee** (an ensemble MCMC algorithm that needs no gradients) with two
different chemistry backends:
- **FastChem backend** -- calls the real FastChem C++ solver (~300 ms/call)
- **Emulator backend** -- calls the transformer neural network (~15 ms/call)

Same sampler, same priors, same mock data. Only the chemistry step differs.
If both posteriors land on the same region of parameter space, the emulator is a
valid and much faster replacement for FastChem.

**Expected runtimes:** emulator chain ~15 min; FastChem chain ~2 hours.

In [ ]:
from jax import config
config.update("jax_enable_x64", True)

import os, sys, time
from pathlib import Path

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from numpy.random import default_rng
import emcee
import corner

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "exojax_demo":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL       = os.environ.get("VULCAN_DEMO_MODEL", "fastchem")
BUNDLE_PATH = (PROJECT_ROOT / "models" / MODEL / "best_exported.npz").resolve()
assert BUNDLE_PATH.exists(), f"bundle not found at {BUNDLE_PATH}"

DIST_ROOT = BUNDLE_PATH.parents[2]
if str(DIST_ROOT) not in sys.path:
    sys.path.insert(0, str(DIST_ROOT))

_STYLE = PROJECT_ROOT / "exojax_demo" / "science.mplstyle"
if _STYLE.exists():
    plt.style.use(str(_STYLE))

SEED       = int(os.environ.get("VULCAN_DEMO_SEED", "20260430"))
N_FIDELITY = 15  # draws for fidelity test; raise for tighter statistics

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"BUNDLE_PATH  : {BUNDLE_PATH}")
print(f"JAX backend  : {jax.default_backend()}  | devices: {jax.devices()}")
print(f"N_FIDELITY   : {N_FIDELITY}")

In [ ]:
from exojax.utils.grids import wavenumber_grid
from exojax.database.exomol.api import MdbExomol
from exojax.opacity import OpaPremodit
from exojax.rt import ArtEmisPure
from exojax.database.contdb import CdbCIA
from exojax.opacity import OpaCIA
from exojax.postproc.specop import SopRotation, SopInstProfile
from exojax.utils.instfunc import resolution_to_gaussian_std
from exojax.atm.atmconvert import vmr_to_mmr
from exojax.database.molinfo.mass import isotope_molmass

nu_grid, wav, resolution = wavenumber_grid(
    22920.0, 23000.0, 1500, unit="AA", xsmode="premodit"
)

mdb = MdbExomol(".database/CO/12C-16O/Li2015", nurange=nu_grid)
opa = OpaPremodit(
    mdb=mdb, nu_grid=nu_grid, auto_trange=[500.0, 2500.0],
    dit_grid_resolution=1.0, allow_32bit=True,
)

NLAYER = 60
art = ArtEmisPure(
    nu_grid=nu_grid, pressure_btm=1.0e1, pressure_top=1.0e-5,
    nlayer=NLAYER, rtsolver="ibased", nstream=8,
)
art.change_temperature_range(300.0, 2500.0)

cdb      = CdbCIA(".database/H2-H2_2011.cia", nurange=nu_grid)
opacia   = OpaCIA(cdb, nu_grid=nu_grid)
sop_rot  = SopRotation(nu_grid, vsini_max=100.0)
sop_inst = SopInstProfile(nu_grid, vrmax=1000.0)

RES_INST  = 70000.0
BETA_INST = resolution_to_gaussian_std(RES_INST)
U1, U2   = 0.0, 0.0
nu_obs   = np.asarray(nu_grid[::5][:-50])
MOLMASS_CO = isotope_molmass("12C-16O")

print(f"CO band R = {resolution:.0f} | nu_grid = {len(nu_grid)} pts | nu_obs = {len(nu_obs)} pts")
print(f"NLAYER = {NLAYER} | P = [{art.pressure.min():.2e}, {art.pressure.max():.2e}] bar")

In [ ]:
from src.constants import SOLAR_ABUNDANCES
from src.models.standalone_inference import guillot_temperature, load_model, make_fastchem_vmr_fn
from src.models.classical_reference import (
    build_exogibbs_element_vector,
    chemsetup_matched_to_fastchem,
    resolve_vulcan_source_root,
    run_fastchem_online,
)
from exogibbs.api.equilibrium import EquilibriumOptions, equilibrium_profile

bundle     = load_model(BUNDLE_PATH)
vmr_fn, species_labels = make_fastchem_vmr_fn(bundle, pressure_order="top_to_bottom")
FASTCHEM_SOURCE_ROOT   = resolve_vulcan_source_root(bundle.config, project_root=PROJECT_ROOT)

# CRITICAL: if any global was frozen during training, gradients for that dimension
# are structurally zero and gradient-based sampling will silently fail.
assert bundle.fixed_globals == {}, (
    f"FAIL: bundle.fixed_globals = {bundle.fixed_globals!r}\n"
    "Re-export from a training run with no fixed globals."
)

chem    = chemsetup_matched_to_fastchem(FASTCHEM_SOURCE_ROOT)
EG_OPTS = EquilibriumOptions(epsilon_crit=1e-11, max_iter=1000, method="vmap_cold")

IDX_CO  = species_labels.index("CO")
IDX_H2  = species_labels.index("H2")
IDX_H2O = species_labels.index("H2O")

MOLAR_MASS = {
    "H2": 2.016, "He": 4.003, "H": 1.008, "O": 15.999, "OH": 17.007,
    "H2O": 18.015, "CO": 28.010, "CO2": 44.009, "CH4": 16.043, "N2": 28.014,
    "NH3": 17.031, "H2S": 34.081, "SH": 33.073, "S": 32.065, "SO": 48.064,
    "SO2": 64.064, "S2": 64.130,
}
MASS_VEC = jnp.array([MOLAR_MASS[s] for s in species_labels], dtype=jnp.float64)

print(f"bundle chemistry : {bundle.chemistry_type} | model : {bundle.model_type}")
print(f"output species   : {species_labels}")
print(f"fixed_globals    : {bundle.fixed_globals}  <- must be empty for full validation")

## Section 1 -- Fidelity: Does the emulator match FastChem?

We draw 15 random atmospheric conditions from the training distribution.
For each one we run both the emulator and the real FastChem solver, then measure
the mean |log10 error| per species across all pressure layers.

**Why log10?** Mixing ratios span many orders of magnitude (e.g. 1e-2 to 1e-10).
A log10 error of 0.01 means the emulator and FastChem agree to within ~2.3%;
0.1 dex means ~26% disagreement.

**Pass thresholds (conservative -- set well above measurement noise):**
- All-species median: < 0.15 dex
- CO: < 0.05 dex (dominant feature in the CO band)
- H2O: < 0.10 dex

In [ ]:
_eps = 1e-30

SAMPLING = bundle.config["sampling"]
PRIOR_FID = {
    "t_int":      (200.0,  600.0),
    "t_eq":       (1100.0, 2200.0),
    "log_gamma":  (-1.5,   0.5),
    "log10_He_H": (np.log10(SAMPLING["he_frac_range"][0]), np.log10(SAMPLING["he_frac_range"][1])),
    "log10_C_H":  (np.log10(SAMPLING["c_frac_range"][0]),  np.log10(SAMPLING["c_frac_range"][1])),
    "log10_O_H":  (np.log10(SAMPLING["o_frac_range"][0]),  np.log10(SAMPLING["o_frac_range"][1])),
    "log10_N_H":  (np.log10(SAMPLING["n_frac_range"][0]),  np.log10(SAMPLING["n_frac_range"][1])),
    "log10_S_H":  (np.log10(SAMPLING["s_frac_range"][0]),  np.log10(SAMPLING["s_frac_range"][1])),
}

def _lhs_uniform(rng, n, lo, hi):
    """Latin-hypercube sampling: ensures even coverage of [lo, hi]."""
    edges = (np.arange(n) + rng.uniform(0.0, 1.0, n)) / n
    rng.shuffle(edges)
    return lo + (hi - lo) * edges

def make_draws(n, seed=SEED):
    rng  = default_rng(seed)
    keys = list(PRIOR_FID.keys())
    cols = {k: _lhs_uniform(rng, n * 3, *PRIOR_FID[k]) for k in keys}
    draws, idx = [], 0
    while len(draws) < n and idx < n * 3:
        d = {k: float(cols[k][idx]) for k in keys}
        idx += 1
        Tp = np.asarray(guillot_temperature(
            art.pressure, t_int_k=d["t_int"], t_eq_k=d["t_eq"],
            log10_delta=0.0, log10_gamma=d["log_gamma"],
        ))
        if Tp.min() < 1.0 or Tp.max() > 3000.0:
            continue
        d["Tarr"]    = Tp
        d["globals"] = {
            "He_H": float(10.0 ** d["log10_He_H"]),
            "C_H":  float(10.0 ** d["log10_C_H"]),
            "O_H":  float(10.0 ** d["log10_O_H"]),
            "N_H":  float(10.0 ** d["log10_N_H"]),
            "S_H":  float(10.0 ** d["log10_S_H"]),
        }
        draws.append(d)
    return draws[:n]

DRAWS = make_draws(N_FIDELITY)

@jax.jit
def _vmr_emul_jit(Tarr, P, g):
    return vmr_fn(Tarr, P, g)

# warmup compile
_w = _vmr_emul_jit(jnp.asarray(DRAWS[0]["Tarr"]), art.pressure,
                   {k: jnp.asarray(v, dtype=jnp.float64) for k, v in DRAWS[0]["globals"].items()})
_w.block_until_ready()

vmr_emul   = np.empty((N_FIDELITY, NLAYER, len(species_labels)))
vmr_fchem  = np.empty_like(vmr_emul)
mae_per_sp = np.empty((N_FIDELITY, len(species_labels)))

t0 = time.perf_counter()
for i, d in enumerate(DRAWS):
    Tj = jnp.asarray(d["Tarr"])
    gj = {k: jnp.asarray(v, dtype=jnp.float64) for k, v in d["globals"].items()}
    vmr_emul[i]  = np.asarray(_vmr_emul_jit(Tj, art.pressure, gj))
    vmr_fchem[i] = np.asarray(run_fastchem_online(
        FASTCHEM_SOURCE_ROOT, np.asarray(art.pressure), d["Tarr"],
        d["globals"], species_labels, bundle.config
    ))
    la = np.log10(np.clip(vmr_emul[i],  _eps, None))
    lb = np.log10(np.clip(vmr_fchem[i], _eps, None))
    mae_per_sp[i] = np.mean(np.abs(la - lb), axis=0)
    if (i + 1) % 5 == 0 or i == N_FIDELITY - 1:
        elapsed = time.perf_counter() - t0
        print(f"  draw {i+1:2d}/{N_FIDELITY}  all-sp MAE = {mae_per_sp[i].mean():.4f} dex  "
              f"({elapsed:.1f}s elapsed)")

mae_all_draws  = mae_per_sp.mean(axis=1)
median_mae_all = float(np.median(mae_all_draws))
med_sp         = np.median(mae_per_sp, axis=0)
print(f"\nSummary over {N_FIDELITY} draws:")
print(f"  All-species median MAE : {median_mae_all:.4f} dex")
print(f"  CO  median MAE         : {med_sp[IDX_CO]:.4f} dex")
print(f"  H2  median MAE         : {med_sp[IDX_H2]:.4f} dex")
print(f"  H2O median MAE         : {med_sp[IDX_H2O]:.4f} dex")

In [ ]:
THRESH_ALL = 0.15
THRESH_CO  = 0.05
THRESH_H2O = 0.10

sort_idx   = np.argsort(med_sp)[::-1]
bar_colors = [
    '#d6604d' if species_labels[j] in ('H2O','CO','CO2','CH4') else '#5aac44'
    for j in sort_idx
]

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(range(len(species_labels)), med_sp[sort_idx],
       color=bar_colors, alpha=0.85, edgecolor='k', lw=0.3)
ax.set_xticks(range(len(species_labels)))
ax.set_xticklabels([species_labels[j] for j in sort_idx], rotation=45, ha='right')
ax.set_ylabel("|log10 MAE| (dex)")
ax.set_title(f"Emulator vs FastChem VMR agreement ({N_FIDELITY} in-distribution draws)")
ax.axhline(THRESH_ALL, ls='--', color='k', alpha=0.45, lw=1.2,
           label=f'all-species threshold ({THRESH_ALL} dex)')
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

pass_all = median_mae_all          < THRESH_ALL
pass_co  = float(med_sp[IDX_CO])  < THRESH_CO
pass_h2o = float(med_sp[IDX_H2O]) < THRESH_H2O
FIDELITY_PASS = pass_all and pass_co and pass_h2o

print("\n=== FIDELITY TEST ===")
print(f"  All-species : {median_mae_all:.4f} dex  "
      f"{'PASS' if pass_all else 'FAIL'}  (threshold {THRESH_ALL})")
print(f"  CO          : {float(med_sp[IDX_CO]):.4f} dex  "
      f"{'PASS' if pass_co else 'FAIL'}  (threshold {THRESH_CO})")
print(f"  H2O         : {float(med_sp[IDX_H2O]):.4f} dex  "
      f"{'PASS' if pass_h2o else 'FAIL'}  (threshold {THRESH_H2O})")

## Section 2 -- Gradients: Does differentiation work?

Gradient-based samplers (e.g. NUTS) are typically 10-100x more efficient than
gradient-free methods. They require that derivatives of the log-likelihood
w.r.t. all parameters are finite and consistent.

FastChem is a C++ subprocess with no derivatives. The emulator is pure JAX, so
gradients flow automatically. This is a key advantage.

**Three checks:**
1. `jax.grad` of a scalar loss w.r.t. log(C/H) -- basic differentiability
2. `jax.jacfwd` vs `jax.jacrev` on the CO profile -- forward and reverse mode AD agree
3. `jax.vmap` over a batch of atmospheres -- required for batched evaluation

In [ ]:
Tarr_ref_np = np.asarray(guillot_temperature(
    art.pressure, t_int_k=400.0, t_eq_k=1500.0, log10_delta=0.0, log10_gamma=-1.0
))
log_C_H_ref = jnp.asarray(np.log10(float(SOLAR_ABUNDANCES["C_H"])), dtype=jnp.float64)

def _globals_from_log_C_H(val):
    return {
        "He_H": jnp.asarray(SOLAR_ABUNDANCES["He_H"], dtype=jnp.float64),
        "C_H":  10.0 ** val,
        "O_H":  jnp.asarray(SOLAR_ABUNDANCES["O_H"], dtype=jnp.float64),
        "N_H":  jnp.asarray(SOLAR_ABUNDANCES["N_H"], dtype=jnp.float64),
        "S_H":  jnp.asarray(SOLAR_ABUNDANCES["S_H"], dtype=jnp.float64),
    }

@jax.jit
def _sum_log10_co(log_C_H_val):
    vmr = vmr_fn(jnp.asarray(Tarr_ref_np), art.pressure, _globals_from_log_C_H(log_C_H_val))
    return jnp.sum(jnp.log10(jnp.clip(vmr[:, IDX_CO], 1e-30)))

@jax.jit
def _log10_co_profile(log_C_H_val):
    vmr = vmr_fn(jnp.asarray(Tarr_ref_np), art.pressure, _globals_from_log_C_H(log_C_H_val))
    return jnp.log10(jnp.clip(vmr[:, IDX_CO], 1e-30))

# Check 1: jax.grad
grad_fn  = jax.jit(jax.grad(_sum_log10_co))
g_scalar = float(grad_fn(log_C_H_ref))
check1   = np.isfinite(g_scalar) and abs(g_scalar) > 1e-6
print(f"Check 1 -- jax.grad  d(sum log10 CO)/d(log C/H):")
print(f"  value = {g_scalar:.4f}  {'PASS -- finite and non-zero' if check1 else 'FAIL'}")

# Check 2: jacfwd vs jacrev
jac_fwd = np.asarray(jax.jacfwd(_log10_co_profile)(log_C_H_ref))
jac_rev = np.asarray(jax.jacrev(_log10_co_profile)(log_C_H_ref))
jac_abs = np.abs(jac_fwd - jac_rev)
jac_rel = jac_abs / (np.abs(jac_fwd).mean() + 1e-12)
check2  = float(jac_abs.max()) < 1e-4 or float(jac_rel.max()) < 1e-3
print(f"\nCheck 2 -- jacfwd vs jacrev on CO(P) profile:")
print(f"  max|fwd-rev| = {jac_abs.max():.2e}  max rel = {jac_rel.max():.2e}  "
      f"{'PASS' if check2 else 'FAIL'}")

# Check 3: vmap
n_batch = min(4, N_FIDELITY)
Tb = jnp.stack([jnp.asarray(DRAWS[i]["Tarr"]) for i in range(n_batch)])
gb = {k: jnp.stack([jnp.asarray(DRAWS[i]["globals"][k], dtype=jnp.float64)
                    for i in range(n_batch)])
      for k in SOLAR_ABUNDANCES}
vmr_batched = jax.vmap(lambda T, g: vmr_fn(T, art.pressure, g))(Tb, gb)
check3 = (
    vmr_batched.shape == (n_batch, NLAYER, len(species_labels))
    and bool(jnp.isfinite(vmr_batched).all())
)
print(f"\nCheck 3 -- jax.vmap over {n_batch} atmospheres:")
print(f"  output shape = {vmr_batched.shape}  {'PASS' if check3 else 'FAIL'}")

GRADIENT_PASS = check1 and check2 and check3
print(f"\n=== GRADIENT TEST ===")
print(f"  Overall: {'PASS -- gradients flow correctly' if GRADIENT_PASS else 'FAIL -- see above'}")

## Section 3 -- Speed: How much faster is the emulator?

Every MCMC step requires one forward-model call per walker.
With 14 walkers and 1500 steps that is 21,000 calls total.
The speedup over FastChem directly determines how long the retrieval takes.

Three backends are timed:
- **FastChem** -- C++ binary launched as a subprocess (no GPU, no JIT)
- **ExoGibbs** -- differentiable Gibbs minimizer in JAX (JIT-compiled)
- **Emulator** -- transformer in JAX (JIT-compiled, trained on FastChem)

Note: ExoGibbs uses different thermodynamic polynomials than FastChem (5-term logK
vs NASA-9). This causes H2O errors of 0.2-1.3 dex near C/O = 0.8.
Section 4 uses FastChem directly -- not ExoGibbs -- as the classical baseline.

In [ ]:
probe  = DRAWS[0]
Tp_j   = jnp.asarray(probe["Tarr"])
gp_j   = {k: jnp.asarray(v, dtype=jnp.float64) for k, v in probe["globals"].items()}
ev_eg  = build_exogibbs_element_vector(chem, probe["globals"], mode="fastchem_proxy")

def _bench(fn, n=5):
    out = fn()
    if hasattr(out, "block_until_ready"):
        out.block_until_ready()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        out = fn()
        if hasattr(out, "block_until_ready"):
            out.block_until_ready()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts)) * 1e3

t_emul = _bench(lambda: _vmr_emul_jit(Tp_j, art.pressure, gp_j), n=10)

t0 = time.perf_counter()
for _ in range(3):
    run_fastchem_online(FASTCHEM_SOURCE_ROOT, np.asarray(art.pressure),
                        probe["Tarr"], probe["globals"], species_labels, bundle.config)
t_fc = (time.perf_counter() - t0) / 3 * 1e3

def _eg():
    return equilibrium_profile(chem, np.asarray(probe["Tarr"]),
                                np.asarray(art.pressure), ev_eg, Pref=1.0, options=EG_OPTS)
_ = _eg()
ts_eg = []
for _ in range(3):
    t0 = time.perf_counter(); _eg(); ts_eg.append(time.perf_counter() - t0)
t_eg = float(np.median(ts_eg)) * 1e3

print("Forward chemistry timing (ms/call, median):")
print(f"  FastChem  (subprocess) : {t_fc:8.1f} ms")
print(f"  ExoGibbs  (JIT-hot)    : {t_eg:8.1f} ms")
print(f"  Emulator  (JIT-hot)    : {t_emul:8.1f} ms")
print(f"\nSpeedup vs FastChem:")
print(f"  ExoGibbs : {t_fc/max(t_eg,0.001):6.1f}x")
print(f"  Emulator : {t_fc/max(t_emul,0.001):6.1f}x")

SPEED_PASS = t_fc / max(t_emul, 0.001) > 10.0
print(f"\n=== SPEED TEST ===")
print(f"  Emulator speedup : {t_fc/max(t_emul,0.001):.0f}x  "
      f"{'PASS' if SPEED_PASS else 'FAIL'}  (threshold: >10x)")

est_emul_min = 14 * 1500 * t_emul / 1e3 / 60
est_fc_min   = 14 * 1500 * t_fc   / 1e3 / 60
print(f"\nEstimated retrieval chain time (14 walkers, 1500 steps):")
print(f"  Emulator : ~{est_emul_min:.0f} min")
print(f"  FastChem : ~{est_fc_min:.0f} min")

## Section 4 -- Retrieval Benchmark: Same answer from both?

### Setup

We generate a synthetic spectrum by running FastChem at known truth parameters,
then add realistic instrument noise. Both backends fit this same dataset.

We run emcee with the same configuration for both:
- **14 walkers** (2 x number of free parameters -- the minimum for emcee)
- **1500 steps** total, first 500 discarded as burn-in
- **14,000 posterior samples** for the corner plot

The 7 free parameters and their uniform prior bounds:

| Parameter | Truth | Prior range | Meaning |
|-----------|-------|-------------|---------|
| `t_int` | 400 K | [200, 600] K | Planet's internal heat -- sets the deep temperature |
| `t_eq` | 1500 K | [1100, 2200] K | Stellar irradiation -- controls upper atmosphere |
| `log_gamma` | -1.0 | [-1.5, 0.5] | Ratio of visible to IR opacity (shapes the PT profile) |
| `logg` | 4.39 | [4.0, 5.0] | log10 surface gravity (cm/s2) |
| `RV` | 40 km/s | [35, 45] | Planet radial velocity (shifts spectral lines) |
| `vsini` | 10 km/s | [5, 15] | Rotational broadening |
| `logZ` | 0.0 | [-1, 1] | log10 metallicity -- scales C/H and O/H together |

The truth is solar composition (logZ = 0), a clean case away from the CO/H2O
chemical transition where both backends are expected to agree well.

In [ ]:
TRUTH = {"t_int":400.0, "t_eq":1500.0, "log_gamma":-1.0,
         "logg":4.39, "RV":40.0, "vsini":10.0, "logZ":0.0}
PARAM_NAMES  = ["t_int","t_eq","log_gamma","logg","RV","vsini","logZ"]
PARAM_LABELS = [r"$T_{\rm int}$", r"$T_{\rm eq}$", r"$\log\gamma$",
                r"$\log g$", "RV", r"$v\sin i$", r"$\log Z$"]
NDIM      = 7
NOISE     = 500.0
PRIOR_LO  = np.array([200.0, 1100.0, -1.5, 4.0, 35.0,  5.0, -1.0])
PRIOR_HI  = np.array([600.0, 2200.0,  0.5, 5.0, 45.0, 15.0,  2.0])  # logZ widened to +2 to match training box (0.1x-100x solar)
TRUTH_ARR = np.array([TRUTH[k] for k in PARAM_NAMES])

def _globals_np(logZ):
    s = 10.0 ** float(logZ)
    return {"He_H": float(SOLAR_ABUNDANCES["He_H"]),
            "C_H":  float(SOLAR_ABUNDANCES["C_H"]) * s,
            "O_H":  float(SOLAR_ABUNDANCES["O_H"]) * s,
            "N_H":  float(SOLAR_ABUNDANCES["N_H"]),
            "S_H":  float(SOLAR_ABUNDANCES["S_H"])}

def _globals_jax(logZ):
    s = 10.0 ** logZ
    return {"He_H": jnp.asarray(SOLAR_ABUNDANCES["He_H"], dtype=jnp.float64),
            "C_H":  jnp.asarray(SOLAR_ABUNDANCES["C_H"],  dtype=jnp.float64) * s,
            "O_H":  jnp.asarray(SOLAR_ABUNDANCES["O_H"],  dtype=jnp.float64) * s,
            "N_H":  jnp.asarray(SOLAR_ABUNDANCES["N_H"],  dtype=jnp.float64),
            "S_H":  jnp.asarray(SOLAR_ABUNDANCES["S_H"],  dtype=jnp.float64)}

def _spec_core(Tarr, grav_cgs, RV, vsini, vmr_table):
    vmr_co = vmr_table[:, IDX_CO]
    vmr_h2 = vmr_table[:, IDX_H2]
    mmw    = jnp.sum(vmr_table * MASS_VEC, axis=-1)
    mmr_co = vmr_to_mmr(vmr_co, MOLMASS_CO, mmw)
    xs_co  = opa.xsmatrix(Tarr, art.pressure)
    dtau_co  = art.opacity_profile_xs(xs_co, mmr_co, MOLMASS_CO, grav_cgs)
    logacia  = opacia.logacia_matrix(Tarr)
    dtau_cia = art.opacity_profile_cia(logacia, Tarr, vmr_h2, vmr_h2, mmw[:,None], grav_cgs)
    F = art.run(dtau_co + dtau_cia, Tarr)
    F = sop_rot.rigid_rotation(F, vsini, U1, U2)
    F = sop_inst.ipgauss(F, BETA_INST)
    return sop_inst.sampling(F, RV, nu_obs)

@jax.jit
def _rt_jit(Tarr, grav_cgs, RV, vsini, vmr_table):
    """JIT-compiled RT -- used inside log_prob_fastchem."""
    return _spec_core(Tarr, grav_cgs, RV, vsini, vmr_table)

@jax.jit
def fspec_emul(t_int, t_eq, log_gamma, logg, RV, vsini, logZ):
    """Full emulator forward model: Guillot PT + emulator chemistry + ExoJAX RT."""
    Tarr = guillot_temperature(art.pressure, t_int_k=t_int, t_eq_k=t_eq,
                                log10_delta=0.0, log10_gamma=log_gamma)
    vmr  = vmr_fn(Tarr, art.pressure, _globals_jax(logZ))
    return _spec_core(Tarr, 10.0**logg, RV, vsini, vmr)

# warmup compile
_ = fspec_emul(*[float(TRUTH[k]) for k in PARAM_NAMES]).block_until_ready()

# generate FastChem mock
Tarr_truth = np.asarray(guillot_temperature(
    art.pressure, t_int_k=TRUTH["t_int"], t_eq_k=TRUTH["t_eq"],
    log10_delta=0.0, log10_gamma=TRUTH["log_gamma"]
))
vmr_truth = np.asarray(run_fastchem_online(
    FASTCHEM_SOURCE_ROOT, np.asarray(art.pressure), Tarr_truth,
    _globals_np(TRUTH["logZ"]), species_labels, bundle.config
))
mu_truth = np.asarray(_rt_jit(
    jnp.asarray(Tarr_truth), 10.0**TRUTH["logg"],
    TRUTH["RV"], TRUTH["vsini"], jnp.asarray(vmr_truth)
))
Fobs = mu_truth + default_rng(SEED + 1).normal(0.0, NOISE, size=len(nu_obs))

print(f"Mock spectrum: {len(Fobs)} pixels | SNR ~ {np.mean(np.abs(mu_truth))/NOISE:.0f}")

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(1e8/nu_obs, Fobs,     lw=0.5, alpha=0.55, label="Mock observation (truth + noise)")
ax.plot(1e8/nu_obs, mu_truth, lw=0.9, label="Truth spectrum (FastChem, no noise)")
ax.set_xlabel(r"Wavelength ($\AA$)")
ax.set_ylabel(r"Flux")
ax.set_title("Mock observation generated by FastChem at truth parameters")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
def log_prob_emulator(theta):
    """Emulator forward model log posterior."""
    if np.any(theta < PRIOR_LO) or np.any(theta > PRIOR_HI):
        return -np.inf
    t_int, t_eq, log_gamma, logg, RV, vsini, logZ = theta
    try:
        spec = np.asarray(fspec_emul(
            float(t_int), float(t_eq), float(log_gamma),
            float(logg), float(RV), float(vsini), float(logZ)
        ))
    except Exception:
        return -np.inf
    if not np.isfinite(spec).all():
        return -np.inf
    return -0.5 * float(np.sum((spec - Fobs)**2)) / (NOISE**2)


def log_prob_fastchem(theta):
    """FastChem forward model log posterior.

    Identical to log_prob_emulator in every way except the chemistry step:
    this calls the real FastChem subprocess instead of the neural network.
    """
    if np.any(theta < PRIOR_LO) or np.any(theta > PRIOR_HI):
        return -np.inf
    t_int, t_eq, log_gamma, logg, RV, vsini, logZ = theta
    Tarr_np = np.asarray(guillot_temperature(
        art.pressure, t_int_k=float(t_int), t_eq_k=float(t_eq),
        log10_delta=0.0, log10_gamma=float(log_gamma)
    ))
    if Tarr_np.min() <= 0 or Tarr_np.max() > 3000:
        return -np.inf
    try:
        vmr_np = np.asarray(run_fastchem_online(
            FASTCHEM_SOURCE_ROOT, np.asarray(art.pressure), Tarr_np,
            _globals_np(float(logZ)), species_labels, bundle.config
        ))
    except Exception:
        return -np.inf
    if not np.isfinite(vmr_np).all():
        return -np.inf
    try:
        spec = np.asarray(_rt_jit(
            jnp.asarray(Tarr_np), 10.0**float(logg),
            float(RV), float(vsini), jnp.asarray(vmr_np)
        ))
    except Exception:
        return -np.inf
    if not np.isfinite(spec).all():
        return -np.inf
    return -0.5 * float(np.sum((spec - Fobs)**2)) / (NOISE**2)


lp_emul = log_prob_emulator(TRUTH_ARR)
lp_fc   = log_prob_fastchem(TRUTH_ARR)
print(f"log_prob at truth:  emulator = {lp_emul:.1f}  |  FastChem = {lp_fc:.1f}")
print("(should be similar -- both fitting the same FastChem-generated mock)")

### Run 1: Emulator + emcee

14 walkers x 1500 steps (500 burn-in + 1000 production = 14,000 posterior samples).
The walkers start in a tight ball near the truth parameters, mimicking what you
would do in practice after a quick MAP optimization.

In [ ]:
NWALKERS = 14    # 2 x NDIM -- the minimum for emcee
NSTEPS   = 1500
NBURNIN  = 500   # first 500 steps discarded; chain should be well-mixed by then

P0       = TRUTH_ARR + np.array([20.0, 100.0, 0.1, 0.05, 1.0, 1.0, 0.1])
P0_SIGMA = np.array([8.0, 40.0, 0.04, 0.02, 0.4, 0.4, 0.04])
rng_init = default_rng(SEED + 10)
pos_emul = np.clip(
    P0 + P0_SIGMA * rng_init.standard_normal((NWALKERS, NDIM)),
    PRIOR_LO, PRIOR_HI
)

print(f"Emulator + emcee: {NWALKERS} walkers x {NSTEPS} steps")
print(f"Production samples: {NWALKERS} x {NSTEPS-NBURNIN} = {NWALKERS*(NSTEPS-NBURNIN):,}")

sampler_emul = emcee.EnsembleSampler(NWALKERS, NDIM, log_prob_emulator)
t0_emul = time.perf_counter()
sampler_emul.run_mcmc(pos_emul, NSTEPS, progress=True)
t_emul_run = time.perf_counter() - t0_emul

samples_emul = sampler_emul.get_chain(discard=NBURNIN, flat=True)
acc_emul     = float(np.mean(sampler_emul.acceptance_fraction))
print(f"\nDone in {t_emul_run/60:.1f} min | acceptance = {acc_emul:.3f} (healthy: 0.2-0.5)")
print(f"Posterior samples shape: {samples_emul.shape}")

### Run 2: FastChem + emcee  *(slow -- expected ~2 hours)*

Identical configuration -- same NWALKERS, NSTEPS, priors, and starting positions.
Only the chemistry step inside `log_prob_fastchem` changes: the real FastChem
solver is called instead of the neural network.

If the two posteriors overlap, the emulator is a valid drop-in replacement.

In [ ]:
pos_fc = np.clip(
    P0 + P0_SIGMA * default_rng(SEED + 20).standard_normal((NWALKERS, NDIM)),
    PRIOR_LO, PRIOR_HI
)

est_min = NWALKERS * NSTEPS * t_fc / 1e3 / 60
print(f"FastChem + emcee: {NWALKERS} walkers x {NSTEPS} steps")
print(f"Estimated runtime: ~{est_min:.0f} min (based on {t_fc:.0f} ms/FastChem call)")

sampler_fc = emcee.EnsembleSampler(NWALKERS, NDIM, log_prob_fastchem)
t0_fc = time.perf_counter()
sampler_fc.run_mcmc(pos_fc, NSTEPS, progress=True)
t_fc_run = time.perf_counter() - t0_fc

samples_fc = sampler_fc.get_chain(discard=NBURNIN, flat=True)
acc_fc     = float(np.mean(sampler_fc.acceptance_fraction))
print(f"\nDone in {t_fc_run/60:.1f} min | acceptance = {acc_fc:.3f} (healthy: 0.2-0.5)")
print(f"Retrieval speedup: {t_fc_run/max(t_emul_run,0.001):.0f}x faster with emulator")

In [ ]:
fig = corner.corner(
    samples_emul,
    labels=PARAM_LABELS, truths=TRUTH_ARR,
    color='#2166ac', alpha=0.85,
    bins=30, plot_datapoints=False,
    fill_contours=True, levels=(0.68, 0.95),
    hist_kwargs=dict(density=True),
    truth_color='k',
    label_kwargs=dict(fontsize=9),
)
corner.corner(
    samples_fc,
    labels=PARAM_LABELS, truths=TRUTH_ARR,
    color='#d6604d', alpha=0.55,
    bins=30, plot_datapoints=False,
    fill_contours=True, levels=(0.68, 0.95),
    hist_kwargs=dict(density=True),
    fig=fig,
)

legend_elems = [
    mlines.Line2D([],[],color='#2166ac',lw=3,label=f'Emulator ({t_emul_run/60:.0f} min)'),
    mlines.Line2D([],[],color='#d6604d',lw=3,label=f'FastChem ({t_fc_run/60:.0f} min)'),
    mlines.Line2D([],[],color='k',lw=2,ls='--',label='Truth'),
]
axarr = np.array(fig.axes).reshape((NDIM, NDIM))
axarr[0, NDIM-1].legend(handles=legend_elems, loc='upper right', fontsize=9)
fig.suptitle("Emulator vs FastChem posterior recovery\n"
             "Same sampler (emcee), same priors, same mock data", y=1.01)
plt.tight_layout(); plt.show()

print(f"\n{'Parameter':<12} {'Truth':>8} {'Emul med':>10} {'FC med':>10} {'|diff|':>8}")
print('-' * 56)
med_emul = np.median(samples_emul, axis=0)
med_fc   = np.median(samples_fc,   axis=0)
for j, (name, tr) in enumerate(zip(PARAM_NAMES, TRUTH_ARR)):
    print(f"{name:<12} {tr:>8.3f} {med_emul[j]:>10.3f} {med_fc[j]:>10.3f} "
          f"{abs(med_emul[j]-med_fc[j]):>8.4f}")

## Section 5 -- Verdict

For each test we compute a simple overlap score:
what fraction of the emulator's 1-sigma interval (16th-84th percentile)
overlaps with FastChem's 1-sigma interval?
A score above 0.3 means the two posteriors are broadly consistent.

In [ ]:
def _overlap(a, b, col):
    lo_a, hi_a = np.percentile(a[:, col], [16, 84])
    lo_b, hi_b = np.percentile(b[:, col], [16, 84])
    overlap = min(hi_a, hi_b) - max(lo_a, lo_b)
    denom   = ((hi_a - lo_a) + (hi_b - lo_b)) / 2
    return max(0.0, overlap / max(denom, 1e-12))

overlap_scores = [_overlap(samples_emul, samples_fc, j) for j in range(NDIM)]
mean_overlap   = float(np.mean(overlap_scores))
RETRIEVAL_PASS = mean_overlap > 0.3

print("Retrieval 1-sigma overlap per parameter:")
for j, name in enumerate(PARAM_NAMES):
    flag = "ok" if overlap_scores[j] > 0.3 else "low"
    print(f"  {name:<12} : {overlap_scores[j]:.2f}  [{flag}]")

print()
print("=" * 62)
print("EMULATOR VALIDATION SUMMARY")
print("=" * 62)

results = [
    ("Fidelity",  FIDELITY_PASS,
     f"all-sp={median_mae_all:.3f}, CO={float(med_sp[IDX_CO]):.3f}, H2O={float(med_sp[IDX_H2O]):.3f} dex"),
    ("Gradients", GRADIENT_PASS,
     "jax.grad finite, jacfwd~jacrev, vmap works"),
    ("Speed",     SPEED_PASS,
     f"emulator {t_fc/max(t_emul,0.001):.0f}x faster than FastChem"),
    ("Retrieval", RETRIEVAL_PASS,
     f"mean 1-sigma overlap = {mean_overlap:.2f} (threshold 0.30)"),
]
for name, passed, detail in results:
    mark = "PASS" if passed else "FAIL"
    print(f"  {name:<12} : {mark:<4}  ({detail})")

overall = all(p for _, p, _ in results)
print("=" * 62)
if overall:
    print("  OVERALL : PASS -- emulator is ready to replace FastChem")
else:
    print("  OVERALL : FAIL -- see items above")
print("=" * 62)

print(f"\nRetrieval wall-clock:")
print(f"  Emulator chain : {t_emul_run/60:.1f} min")
print(f"  FastChem chain : {t_fc_run/60:.1f} min")
print(f"  Speedup        : {t_fc_run/max(t_emul_run,0.001):.0f}x")

## Save results to disk

Writes a timestamped directory under `diagnostics/` containing:
- `summary.json` -- all PASS/FAIL flags, metrics, timing, and retrieved parameter medians
- `posteriors.npz` -- raw emcee samples from both backends (shape: 14000 x 7)
- `fidelity.npz` -- per-draw per-species VMR MAE arrays

In [ ]:
import datetime
from pathlib import Path

OUTDIR = Path("diagnostics") / (
    "validation_"
    + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
OUTDIR.mkdir(parents=True, exist_ok=True)
print("saving to:", OUTDIR.resolve())

# -- summary.json --------------------------------------------------------
import json as _json

def _interval(arr, col):
    lo, hi = np.percentile(arr[:, col], [16, 84])
    return {"lo": float(lo), "median": float(np.median(arr[:, col])), "hi": float(hi)}

summary = {
    "tests": {
        "fidelity":  {"pass": bool(FIDELITY_PASS),
                       "all_species_median_dex": float(median_mae_all),
                       "co_median_dex":          float(med_sp[IDX_CO]),
                       "h2o_median_dex":         float(med_sp[IDX_H2O])},
        "gradients": {"pass": bool(GRADIENT_PASS)},
        "speed":     {"pass": bool(SPEED_PASS),
                       "emulator_ms":  float(t_emul),
                       "fastchem_ms":  float(t_fc),
                       "speedup":      float(t_fc / max(t_emul, 0.001))},
        "retrieval": {"pass": bool(RETRIEVAL_PASS),
                       "mean_overlap": float(mean_overlap),
                       "overlap_per_param": {
                           name: float(overlap_scores[j])
                           for j, name in enumerate(PARAM_NAMES)}},
    },
    "overall_pass": all(r[1] for r in results),
    "truth": {k: float(v) for k, v in zip(PARAM_NAMES, TRUTH_ARR)},
    "emulator_chain": {
        "wall_min":       float(t_emul_run / 60),
        "acceptance":     float(acc_emul),
        "n_samples":      int(samples_emul.shape[0]),
        "medians":        {name: _interval(samples_emul, j)
                            for j, name in enumerate(PARAM_NAMES)},
    },
    "fastchem_chain": {
        "wall_min":       float(t_fc_run / 60),
        "acceptance":     float(acc_fc),
        "n_samples":      int(samples_fc.shape[0]),
        "medians":        {name: _interval(samples_fc, j)
                            for j, name in enumerate(PARAM_NAMES)},
    },
}

with open(OUTDIR / "summary.json", "w") as f:
    _json.dump(summary, f, indent=2)
print("wrote summary.json")

# -- posteriors.npz ------------------------------------------------------
np.savez_compressed(
    OUTDIR / "posteriors.npz",
    samples_emulator=samples_emul,
    samples_fastchem=samples_fc,
    param_names=np.array(PARAM_NAMES),
    truth=TRUTH_ARR,
    prior_lo=PRIOR_LO,
    prior_hi=PRIOR_HI,
)
print("wrote posteriors.npz  ", samples_emul.shape, samples_fc.shape)

# -- fidelity.npz --------------------------------------------------------
np.savez_compressed(
    OUTDIR / "fidelity.npz",
    mae_per_draw_per_species=mae_per_sp,
    median_mae_per_species=med_sp,
    species_labels=np.array(species_labels),
    vmr_emulator=vmr_emul,
    vmr_fastchem=vmr_fchem,
)
print("wrote fidelity.npz")

print("\nAll results saved to:", OUTDIR)